# Rebuilding Green Carbon Wealth from raw sources

**Audience.** Researcher rebuilding BNC2 with a transparent country-level data pipeline.

**Learning goals.** By the end, you can identify the raw sources, choose an estimator explicitly, calculate annual GCW, and inspect the outputs before using them in the manuscript.

**Rule.** This notebook orchestrates reusable code in `src/gcw/`. The scientific calculation stays in tested modules rather than hidden in notebook cells.


## Outline

1. Define the accounting object and the required sources.
2. Check whether all inputs for each estimator are available.
3. Run a documented forest-flow estimator.
4. Value uptake with the BCW GSCC and merge with BCW only after overlap checks.
5. Read validation outputs and record the method decision.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from gcw.config import GSCC_2015_USD_PER_TCO2, INTERIM, OUTPUT, RAW
from gcw.valuation import carbon_to_co2, value_uptake

print(f'Repository root: {ROOT}')
print(f'Working GSCC: {GSCC_2015_USD_PER_TCO2} constant 2015 US$/tCO2')
print('One tonne of carbon equals', carbon_to_co2(1.0), 'tonnes of CO2.')


## Step 1 — Define the quantities before loading data

The main result is an annual welfare contribution:

\[GCW_i = F_i^C \times 44/12 \times GSCC.\]

`F_i^C` must be a documented annual uptake in tonnes of carbon. A carbon stock, an additional mitigation potential, a mitigation cost and a finance need are different quantities.


In [ ]:
registry = pd.read_csv(ROOT / 'data_source' / 'metadata' / 'source_registry.csv')
registry[['source_id', 'raw_target', 'intended_use', 'status']]


## Step 2 — Choose an estimator transparently

- `faostat` is the direct national baseline. It requires a standardized annual FAOSTAT table, with the selected Item, Element, unit and sign convention recorded.
- `pan_area_rate` is a sensitivity estimator. It requires Pan biome rates and forest areas by country, biome and forest class. It cannot be treated as an observed national flux merely because it has national rows.

Run both once their inputs are available. Choose the paper's primary estimator only after comparing coverage, global totals, physical boundaries and uncertainty.


In [ ]:
required_inputs = {
    'faostat': [INTERIM / 'faostat_forest_flux.csv'],
    'pan_area_rate': [INTERIM / 'pan_biome_rates.csv', INTERIM / 'forest_area_by_biome.csv'],
    'ncw_merge': [RAW / 'bcw' / 'country_level_bcw.csv'],
}
for method, paths in required_inputs.items():
    print(method)
    for source_path in paths:
        print('  ', 'READY  ' if source_path.exists() else 'MISSING', source_path.relative_to(ROOT))


## Step 3 — Run the pipeline

Set `METHOD` only after the corresponding raw source has been standardized. The pipeline writes canonical CSVs and validation reports. It writes an NCW table only when the final BCW table is supplied. Do not set an overlap correction to zero merely because the mangrove check has not yet been completed.


In [ ]:
METHOD = 'faostat'  # change to 'pan_area_rate' only after both Pan interim tables exist
RUN_PIPELINE = False  # change to True only when the prerequisite check above is green

if RUN_PIPELINE:
    command = [sys.executable, str(ROOT / 'scripts' / 'run_pipeline.py'), '--method', METHOD]
    bcw_path = RAW / 'bcw' / 'country_level_bcw.csv'
    if bcw_path.exists():
        command.extend(['--bcw', str(bcw_path)])
    subprocess.run(command, check=True)
    gcw = pd.read_csv(OUTPUT / 'country_level_gcw.csv')
    display(gcw.head())
    print('Global GCW:', f"{gcw['gcw_usd_2015_yr'].sum() / 1e12:.3f} trillion 2015 US$/yr")
else:
    print('Pipeline deliberately not run until raw-source prerequisites are complete.')


## Step 4 — Validation before interpretation

Read every file in `output/validation/`. Check country coverage, duplicate ISO codes, missing variables, selected years, sign convention, units and the reconciliation of global totals. Compare FAOSTAT and Pan-area-rate estimates before selecting the main specification.

### Exercise

Write a short method decision in `data_source/reference/method_decision.md` answering:

1. What exact flux boundary does each estimator measure?
2. Which estimator best matches the manuscript's stated estimand?
3. Which global and country-level checks support that choice?
4. Which uncertainty can be reported now, and which remains a placeholder?
